In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from datetime import datetime
import glob
from reprojection import *
from show import *

In [2]:
files = np.array(sorted(glob.glob('/home/ulyanov/data/solo/phi/2026/blos_/*')))
#files = np.array(sorted(glob.glob('C:/Users/ikiru/data/4science/blos/*')))
#files = np.array(sorted(glob.glob('/home/ulyanov/data/sdo/hmi/2315/*')))
dates = np.array([datetime.fromisoformat(file.split('/')[-1].split('_')[-3]) for file in files])

#t = np.where(np.all([dates > datetime(2026,7,22),
#                     dates < datetime(2026,8,30)], axis=0))[0] # CR2314

#t = np.where(np.all([dates > datetime(2026,8,30),
#                     dates < datetime(2026,9,29)], axis=0))[0] # CR2315


t = np.where(np.all([dates > datetime(2026,7,22),
                     dates < datetime(2026,9,29)], axis=0))[0] # CR2314 + CR2315

files = files[t]
files[0], files[-1], len(files)

(np.str_('/home/ulyanov/data/solo/phi/2026/blos_/phi-fdt-blos_20260722T030009_V202610210936C_0647220501.fits'),
 np.str_('/home/ulyanov/data/solo/phi/2026/blos_/phi-fdt-blos_20260928T213003_V202610202220C_0649280507.fits'),
 215)

In [3]:
data, view = make_map(files[:], correct_dr=False, kind='synoptic', minmu=0.1, wsid=13, sine=False, full_output=True)

/home/ulyanov/PycharmProjects/reprojection/src/reprojection.py:98: RuntimeWarning: invalid value encountered in divide
  w = np.expand_dims(n / N, (-2, -1))


KeyboardInterrupt: 

In [15]:
fig, ax = show_data(data[0], view, cmap='seismic', vmin=-20, vmax=20)

In [16]:
fig, ax = show_data(np.sqrt(data[1]), view, cmap='inferno', vmin=0, vmax=20)

In [17]:
fig, ax = show_data(data[0] / np.sqrt(data[1]), view, cmap='seismic', vmin=-10, vmax=10)

In [18]:
q = data[0] / np.sqrt(data[1])

plt.figure(figsize=(10,8))
plt.plot(np.nanstd(q, axis=1))
plt.tight_layout()

In [19]:
Q = []
wsid_range = np.arange(9, 16, 0.1)
for wsid in wsid_range:
    print(wsid)

    data, view = make_map(files[:], correct_dr=False, kind='synoptic', minmu=0.1, wsid=wsid, sine=False, full_output=True)
    Q += [np.nanstd(data[0] / np.sqrt(data[1]), axis=1)]

Q = np.array(Q)

np.savez('Q.npz', Q=Q)

/tmp/ipykernel_143122/839560839.py:5: RuntimeWarning: invalid value encountered in sqrt
  Q += [np.nanstd(data[0] / np.sqrt(data[1]), axis=1)]


In [23]:
A, B, C = 14.712, -2.396, -1.787

q = np.nanargmax(Q, axis=0)

nx = len(q)
lat = (np.arange(nx) / (nx - 1) * 2 - 1) * 90
y = np.sin(lat * np.pi / 180)

plt.figure(figsize=(10,8))
plt.plot(lat, wsid_range[q])
plt.plot(lat, A + B * y ** 2 + C * y ** 4)
plt.tight_layout()

In [5]:
#np.savez('diff_rot.npz', data=Q, wsid=wsid_range, lat=lat)

In [22]:
plt.figure(figsize=(10,8))
plt.imshow(Q, origin='lower', aspect='auto', cmap='jet', interpolation='none', vmin=0, vmax=10)
plt.tight_layout()

In [35]:
a, b = np.min(data, axis=1), np.max(data, axis=1)

plt.figure(figsize=(10,8))
plt.plot(b - a)
plt.ylim(0,50)
plt.tight_layout()

In [47]:
WSID = 360 / 25.38
A, B, C = 14.712, -2.396, -1.787
RSUN = 696e8

def diff_rot(r, *, a=A-WSID, b=B, c=C, u0=1000, lat0=45, q=0.5):
    def f(x):
        return np.sin(2 * x * np.pi / 180) + q * np.sin(4 * x * np.pi / 180) + (2 * q - 1) / 3 * np.sin(6 * x * np.pi / 180)
    theta, phi = r
    y = np.sin(theta * np.pi / 180)
    u = u0 / RSUN * f(theta) / f(lat0) * 180 / np.pi * 24 * 3600
    return np.array([u, (a + b * y ** 2 + c * y ** 4)])

In [48]:
def step(func, r, h):
    k1 = func(r)
    k2 = func(r + k1 * h / 2)
    k3 = func(r + k2 * h / 2)
    k4 = func(r + k3 * h)
    return r + h / 6 * (k1 + 2 * k2 + 2 * k3 + k4)

def rk4(func, r, h, n):
    out = [r]
    for i in range(n):
        out.append(step(func, out[-1], h))
    return np.array(out)


In [49]:
n = 500

#phi = np.random.uniform(0, 360, size=n)
#phi = np.linspace(0, 360)
phi = np.arange(0, 360, 30)
theta = np.ones_like(phi) * 5
#theta = np.random.normal(25, 15, size=n) * (np.random.binomial(1, 0.5, size=n) * 2 - 1)
#theta = np.random.uniform(-90, 90, size=n)
a = np.array((theta, phi))

q = np.transpose(rk4(diff_rot, a, 1e-1, 10000), (1,0,2))

transform = ~(view.transform() + ToSpherical())
q = np.array(transform(*q, 1)[:-1])

In [50]:
#fig, ax = show_data(data, view, cmap='seismic', vmin=-20, vmax=20)
fig, ax = show_data(data, view, label=r'$B_{r}$, G', cmap='seismic', vmin=-20, vmax=20)
ax.plot(q[1], q[0], color='black', lw=0.7, alpha=0.4)

In [7]:
360 / (365 / 12)

11.835616438356164